In [1]:
import numpy as np
import pandas as pd

In [2]:
print("Loading data...")

Loading data...


In [3]:
df_income = pd.read_csv('../data/raw/01_income_statement_annual.csv')
df_balance = pd.read_csv('../data/raw/02_balance_sheet_annual.csv')
df_cashflow = pd.read_csv('../data/raw/03_cash_flow_annual.csv')
df_stock = pd.read_csv('../data/raw/04_stock_price_annual_history.csv')

In [4]:
df_income['period_ending']   = pd.to_datetime(df_income['period_ending'])
df_balance['period_ending']   = pd.to_datetime(df_balance['period_ending'])
df_cashflow['period_ending']   = pd.to_datetime(df_cashflow['period_ending'])

In [5]:
df_income['fy_num']   = df_income['fiscal_year'].str.extract(r'(\d{4})').astype(int)
df_balance['fy_num']  = df_balance['fiscal_year'].str.extract(r'(\d{4})').astype(int)
df_cashflow['fy_num'] = df_cashflow['fiscal_year'].str.extract(r'(\d{4})').astype(int)

In [6]:
df_income   = df_income.sort_values('fy_num').reset_index(drop=True)
df_balance  = df_balance.sort_values('fy_num').reset_index(drop=True)
df_cashflow = df_cashflow.sort_values('fy_num').reset_index(drop=True)
df_stock    = df_stock.sort_values('year').reset_index(drop=True)

In [7]:
COVID_OUTLIER_FY = 2022
df_income['is_outlier_year']   = df_income['fy_num'] == COVID_OUTLIER_FY
df_balance['is_outlier_year']  = df_balance['fy_num'] == COVID_OUTLIER_FY
df_cashflow['is_outlier_year'] = df_cashflow['fy_num'] == COVID_OUTLIER_FY

In [8]:
df_income_balance = df_income.merge(df_balance, on=['fiscal_year', 'fy_num'], suffixes=('_is', '_bs'))

In [9]:
df_income_balance

,fiscal_year,period_ending_is,revenue_eur_m,revenue_growth_pct,cost_of_revenue_eur_m,gross_profit_eur_m,sg_a_eur_m,depreciation_amortization_eur_m,total_operating_expenses_eur_m,operating_income_eur_m,...,long_term_debt_eur_m,total_liabilities_eur_m,common_stock_eur_m,retained_earnings_eur_m,shareholders_equity_eur_m,total_debt_eur_m,net_cash_debt_eur_m,book_value_per_share_eur,shares_outstanding_m,is_outlier_year_bs
0,FY2022,2022-03-31,4801,193.49,4010,791.1,411.3,719.4,1131,-339.6,...,3715.0,9605,6.8,2881,5545,5077,-1474.0,4.89,1135,True
1,FY2023,2023-03-31,10775,124.44,7735,3040.0,674.4,923.2,1598,1443.0,...,2853.0,10763,6.9,4180,5643,4116,539.3,4.96,1139,False
2,FY2024,2024-03-31,13444,24.77,9566,3877.0,757.2,1060.0,1817,2061.0,...,2532.0,9561,6.9,5900,7614,2747,1366.0,6.68,1140,False
3,FY2025,2025-03-31,13949,3.75,10298,3651.0,878.4,1214.0,2093,1558.0,...,1685.0,10470,6.4,5589,7037,2683,1281.0,6.61,1064,False
4,FY2026,2026-03-31,15544,11.44,10908,4636.0,888.5,1373.0,2262,2374.0,...,147.8,9646,6.3,6778,10101,1492,2054.0,9.68,1044,False


In [10]:
df_cashflow_renamed = df_cashflow.add_suffix('_cf')
df_cashflow_renamed = df_cashflow_renamed.rename(columns={'fy_num_cf': 'fy_num'})
df_full = df_income_balance.merge(df_cashflow_renamed, on='fy_num', how='outer')
df_full = df_full.sort_values('fy_num').reset_index(drop=True)

In [11]:
df_full

,fiscal_year,period_ending_is,revenue_eur_m,revenue_growth_pct,cost_of_revenue_eur_m,gross_profit_eur_m,sg_a_eur_m,depreciation_amortization_eur_m,total_operating_expenses_eur_m,operating_income_eur_m,...,long_term_debt_repaid_eur_m_cf,net_debt_issued_repaid_eur_m_cf,repurchase_of_common_stock_eur_m_cf,common_dividends_paid_eur_m_cf,financing_cash_flow_eur_m_cf,net_cash_flow_eur_m_cf,free_cash_flow_eur_m_cf,free_cash_flow_growth_pct_cf,free_cash_flow_margin_pct_cf,is_outlier_year_cf
0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1027.0,1202.0,0.0,0.0,1623.0,84.3,-2743.0,NaN,-167.67,False
1,FY2022,2022-03-31,4801.0,193.49,4010.0,791.1,411.3,719.4,1131.0,-339.6,...,-1775.0,-583.3,0.0,0.0,-536.5,18.3,758.9,NaN,15.81,True
2,FY2023,2023-03-31,10775.0,124.44,7735.0,3040.0,674.4,923.2,1598.0,1443.0,...,-1086.0,-1086.0,0.0,0.0,-1054.0,930.3,1976.0,160.42,18.34,False
3,FY2024,2024-03-31,13444.0,24.77,9566.0,3877.0,757.2,1060.0,1817.0,2061.0,...,-1143.0,-1143.0,0.0,-199.5,-1326.0,276.1,766.0,-61.24,5.70,False
4,FY2025,2025-03-31,13949.0,3.75,10298.0,3651.0,878.4,1214.0,2093.0,1558.0,...,-86.4,-86.4,-1478.0,-437.7,-1997.0,-12.1,1863.0,143.24,13.36,False
5,FY2026,2026-03-31,15544.0,11.44,10908.0,4636.0,888.5,1373.0,2262.0,2374.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
df_balance['debt_to_equity'] = (df_balance['total_debt_eur_m'] / df_balance['shareholders_equity_eur_m']).round(3)
df_balance['current_ratio'] = (df_balance['total_current_assets_eur_m'] / df_balance['total_current_liabilities_eur_m']).round(3)

In [13]:
df_income_balance['roe_pct'] = (df_income_balance['net_income_eur_m'] / df_income_balance['shareholders_equity_eur_m'] * 100).round(2)

In [14]:
df_stock['price_range_pct'] = ((df_stock['year_high_usd'] - df_stock['year_low_usd']) / df_stock['average_price_usd'] * 100).round(2)

In [15]:
df_clean_income = df_income[df_income['fy_num'] > COVID_OUTLIER_FY].copy()
n_years = df_clean_income['fy_num'].max() - df_clean_income['fy_num'].min()
revenue_start = df_clean_income.loc[df_clean_income['fy_num'].idxmin(), 'revenue_eur_m']
revenue_end = df_clean_income.loc[df_clean_income['fy_num'].idxmax(), 'revenue_eur_m']
revenue_cagr = ((revenue_end / revenue_start) ** (1 / n_years) - 1) * 100

In [16]:
print("\n--- BALANCE SHEET HEALTH RATIOS ---")
print(df_balance[['fiscal_year', 'debt_to_equity', 'current_ratio', 'is_outlier_year']])
 
print("\n--- PROFITABILITY (ROE) BY YEAR ---")
print(df_income_balance[['fiscal_year', 'net_income_eur_m', 'roe_pct']])
 
print(f"\nRevenue CAGR (FY2023-FY2026, excludes COVID-recovery FY2022): {revenue_cagr:.2f}%")


--- BALANCE SHEET HEALTH RATIOS ---
  fiscal_year  debt_to_equity  current_ratio  is_outlier_year
0      FY2022           0.916          1.014             True
1      FY2023           0.729          0.796            False
2      FY2024           0.361          0.910            False
3      FY2025           0.381          0.737            False
4      FY2026           0.148          0.903            False

--- PROFITABILITY (ROE) BY YEAR ---
  fiscal_year  net_income_eur_m  roe_pct
0      FY2022            -240.8    -4.34
1      FY2023            1314.0    23.29
2      FY2024            1917.0    25.18
3      FY2025            1612.0    22.91
4      FY2026            2174.0    21.52

Revenue CAGR (FY2023-FY2026, excludes COVID-recovery FY2022): 12.99%


In [17]:
df_income.to_csv('../data/processed/01_income_statement.csv', index=False)
df_balance.to_csv('../data/processed/02_balance_sheet.csv', index=False)
df_cashflow.to_csv('../data/processed/03_cash_flow.csv', index=False)
df_stock.to_csv('../data/processed/04_stock_price.csv', index=False)
df_income_balance.to_csv('../data/processed/05_income_balance_merged.csv', index=False)

In [18]:
print("\nSuccess! Cleaned data exported to /home/claude/ryanair_data/")


Success! Cleaned data exported to /home/claude/ryanair_data/
